# 🚀 Chapter 4: Vector Applications in Data Science

---

## 4.1 Bridging the Gap: From Theory to Practice

In the previous chapters, we established the fundamental mechanics of vectors: addition, scalar multiplication, the dot product, and the concepts of span and linear independence. While these abstract concepts are beautiful in their own right, the true power of linear algebra is unleashed when we apply these tools to real-world data.

In Data Science, a vector is not just an arrow in space; it is a **data object**. 
- A vector could represent a user's movie ratings.
- It could represent a 1-second snippet of an audio recording.
- It could represent a patient's medical history.

In this chapter, we will explore three foundational applications of vector operations that are used every day in modern machine learning: **Cosine Similarity & Correlation**, **Time Series Filtering (Template Matching)**, and **Distance Calculations for Clustering (k-Means)**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set plotting style for better visualization
%matplotlib inline
np.random.seed(42)
print("Scientific libraries successfully imported for Chapter 4.")

## 4.2 Cosine Similarity and Pearson Correlation

How do we measure if two datasets (vectors) are similar? The **dot product** is our primary tool, but it has a flaw: it is heavily influenced by the magnitude (length) of the vectors. If you double the values in a vector, the dot product doubles, even though the underlying pattern hasn't changed.

**1. Cosine Similarity:**
To fix this, we normalize the dot product by dividing it by the product of the vectors' magnitudes. This isolates the *angle* between the vectors, giving us a pure measure of alignment (ranging from -1 to 1). 

$$ \text{Cosine Similarity} = \cos(\theta) = \frac{a \cdot b}{||a|| \, ||b||} $$

**2. Pearson Correlation Coefficient:**
In statistics, the Pearson correlation coefficient is the gold standard for measuring the linear relationship between two variables. But what is it algebraically? **Pearson correlation is simply the cosine similarity of mean-centered vectors.**

If we subtract the mean of a vector from all its elements, the vector becomes centered at zero. By computing the cosine similarity of these mean-centered vectors, we achieve a metric that is completely invariant to both scale (magnitude) and baseline shifts (mean). This unifies statistics and linear algebra perfectly!

In [ ]:
# Let's create two correlated vectors (e.g., studying hours vs. test scores)
v1 = np.random.rand(50) * 10            # Variable 1: Study hours
v2 = v1 + np.random.normal(0, 2, 50)    # Variable 2: Scores (correlated with noise)

# --- Cosine Similarity ---
cos_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# --- Pearson Correlation (from scratch using Linear Algebra) ---
# 1. Mean-center the vectors
v1_centered = v1 - np.mean(v1)
v2_centered = v2 - np.mean(v2)

# 2. Compute Cosine Similarity on the centered vectors
pearson_corr = np.dot(v1_centered, v2_centered) / (np.linalg.norm(v1_centered) * np.linalg.norm(v2_centered))

# Verify against NumPy's built-in statistics function
numpy_corr = np.corrcoef(v1, v2)[0, 1]

print(f"Cosine Similarity           : {cos_sim:.4f}")
print(f"Pearson Correlation (Manual): {pearson_corr:.4f}")
print(f"Pearson Correlation (NumPy) : {numpy_corr:.4f}")
print("Notice how our manual Linear Algebra approach perfectly matches the statistical formula!")

# Visualization
plt.figure(figsize=(6, 4))
plt.scatter(v1, v2, color='teal', alpha=0.7)
plt.title(f"Scatter Plot of Correlated Variables (r = {pearson_corr:.2f})")
plt.xlabel("Variable 1")
plt.ylabel("Variable 2")
plt.grid(alpha=0.3)
plt.show()

## 4.3 Time Series Filtering (Template Matching)

Another brilliant application of the dot product is finding specific patterns within a larger sequence, such as a time series or an audio signal. 

Imagine we have a long signal vector and a small template vector (often called a **kernel** or **filter**). If we slide this kernel across the signal, computing the dot product at every possible position, the dot product will act as a "similarity detector". 

When the segment of the signal perfectly aligns with the shape of the kernel, the dot product produces a massive spike (highly positive). If they are completely mismatched, the dot product drops. This sliding dot product operation is fundamentally what drives **Convolutional Neural Networks (CNNs)** and signal processing!

In [ ]:
# 1. Create a noisy time series signal (100 time points)
signal = np.random.randn(100)

# Inject a specific pattern (a spike) in the middle of the signal
pattern_location = 40
signal[pattern_location:pattern_location+10] += np.array([0, 1, 3, 5, 8, 5, 3, 1, 0, -1])

# 2. Create the Kernel (The template we are looking for)
kernel = np.array([0, 1, 3, 5, 8, 5, 3, 1, 0, -1])
k_len = len(kernel)

# 3. Perform Template Matching (Sliding Dot Product / Cross-Correlation)
feature_map = np.zeros(len(signal) - k_len + 1)

for i in range(len(feature_map)):
    # Extract the chunk of the signal
    signal_chunk = signal[i : i + k_len]
    # Compute the dot product between the chunk and the kernel
    feature_map[i] = np.dot(signal_chunk, kernel)

# 4. Visualization
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax[0].plot(signal, color='gray', label='Raw Noisy Signal')
ax[0].axvspan(pattern_location, pattern_location+k_len, color='red', alpha=0.2, label='Hidden Pattern')
ax[0].set_title("Original Time Series Signal")
ax[0].legend(loc='upper right')

ax[1].plot(feature_map, color='purple', linewidth=2, label='Dot Product Output')
ax[1].set_title("Feature Map (Result of Sliding Dot Product)")
ax[1].set_xlabel("Time Step")
ax[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

print("Notice how the sliding dot product creates a massive spike exactly where the hidden pattern lives. This is how machines 'see' features!")

## 4.4 Distance Calculations for Clustering (k-Means)

The final application in this chapter looks at vector subtraction and the L2 Norm. 

If we subtract vector $b$ from vector $a$, we get a new vector that points from $b$ to $a$. The length (magnitude) of this difference vector is the exact **Euclidean distance** between the two points in space:

$$ \text{Distance} = ||a - b|| $$

This simple calculation is the entire engine behind the **k-Means Clustering** algorithm. The algorithm simply measures the vector distance from every data point to a set of cluster centroids, and assigns each point to the centroid with the smallest distance.

In [ ]:
# Imagine we have a new customer (a vector) and three existing customer segments (centroids)
customer = np.array([4.0, 5.0])

centroids = np.array([
    [1.0, 1.0],  # Centroid 0: Low spend, low frequency
    [3.0, 4.0],  # Centroid 1: Mid spend, mid frequency
    [7.0, 8.0]   # Centroid 2: High spend, high frequency
])

# Calculate Euclidean distance (L2 norm of the difference) from the customer to each centroid
distances = np.zeros(len(centroids))
for i, centroid in enumerate(centroids):
    distances[i] = np.linalg.norm(customer - centroid)

# Find the index of the minimum distance
closest_cluster = np.argmin(distances)

print(f"Distances to centroids: {np.round(distances, 2)}")
print(f"The customer belongs to Cluster {closest_cluster} because it has the smallest vector distance.")

# Visualization
plt.figure(figsize=(6, 6))
plt.scatter(centroids[:, 0], centroids[:, 1], c=['red', 'green', 'blue'], s=200, marker='X', label='Centroids')
plt.scatter(customer[0], customer[1], c='black', s=100, marker='o', label='New Customer')

# Draw a line to the closest centroid
plt.plot([customer[0], centroids[closest_cluster][0]], 
         [customer[1], centroids[closest_cluster][1]], 'k--', alpha=0.5)

plt.xlim(0, 9)
plt.ylim(0, 9)
plt.title("k-Means Vector Distance Calculation")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 4.5 Chapter Summary

In Chapter 4, we transitioned from abstract math to tangible data science tools:
- **Pearson Correlation** is simply the cosine similarity of mean-centered vectors. The dot product fundamentally measures feature alignment.
- **Time Series Filtering and CNNs** rely on computing sequential dot products between a signal and a kernel to detect local patterns.
- **Clustering Algorithms** like k-Means rely on the L2 norm of vector differences to assign data points to their closest geometric neighborhoods.

Understanding these algebraic roots allows you to look past the "black box" of high-level machine learning libraries and truly grasp how the algorithms are making decisions.